# Preprocessing
Prepare both datasets (Math and Portuguese) for modeling.

Steps:
1. Encode binary categorical variables
2. One-hot encode nominal categorical variables
3. Define targets: `G3` (regression) and `pass` (classification, G3 ≥ 10)
4. Train/test split
5. Save processed datasets

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

math_df = pd.read_csv('./data/student-mat.csv', sep=';')
por_df  = pd.read_csv('./data/student-por.csv', sep=';')

print('Math:', math_df.shape)
print('Portuguese:', por_df.shape)

Math: (395, 33)
Portuguese: (649, 33)


## Encode Binary Categorical Variables

Some columns only have two possible values. We map them to 0/1 so the model can use them as numbers.

| Column | Mapping |
|--------|---------|
| `school` | GP = 0, MS = 1 |
| `sex` | F = 0, M = 1 |
| `address` | R = 0, U = 1 |
| `famsize` | LE3 = 0, GT3 = 1 |
| `Pstatus` | A = 0, T = 1 |
| `schoolsup`, `famsup`, `paid`, `activities`, `nursery`, `higher`, `internet`, `romantic` | no = 0, yes = 1 |

In [2]:
def encode_binary(df):
    df = df.copy()

    df['school'] = df['school'].map({'GP': 0, 'MS': 1})
    df['sex'] = df['sex'].map({'F': 0, 'M': 1})
    df['address']  = df['address'].map({'R': 0, 'U': 1})
    df['famsize']  = df['famsize'].map({'LE3': 0, 'GT3': 1})
    df['Pstatus']  = df['Pstatus'].map({'A': 0, 'T': 1})

    yes_no_cols = ['schoolsup', 'famsup', 'paid', 'activities',
                   'nursery', 'higher', 'internet', 'romantic']
    for col in yes_no_cols:
        df[col] = df[col].map({'no': 0, 'yes': 1})

    return df

math_df = encode_binary(math_df)
por_df  = encode_binary(por_df)

print('Binary encoding done.')
math_df.head(3)

Binary encoding done.


,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,...,famrel,freetime,goout,Dalc,Walc,health,absences,G1,G2,G3
0,0,0,18,1,1,0,4,4,at_home,teacher,...,4,3,4,1,1,3,6,5,6,6
1,0,0,17,1,1,1,1,1,at_home,other,...,5,3,3,1,1,3,4,5,5,6
2,0,0,15,1,0,1,1,1,at_home,other,...,4,3,2,2,3,3,10,7,8,10


## One-Hot Encode Nominal Categorical Variables

Some columns have more than 2 categories with no natural order. So we apply one-hot ecoding.

Columns: `Mjob`, `Fjob`, `reason`, `guardian`

In [3]:
nominal_cols = ['Mjob', 'Fjob', 'reason', 'guardian']

math_df = pd.get_dummies(math_df, columns=nominal_cols, drop_first=True)
por_df  = pd.get_dummies(por_df,  columns=nominal_cols, drop_first=True)

print('Math shape after encoding:', math_df.shape)
print('Portuguese shape after encoding:', por_df.shape)
print('\nNew columns added:')
print([c for c in math_df.columns if any(c.startswith(n) for n in nominal_cols)])

Math shape after encoding: (395, 42)
Portuguese shape after encoding: (649, 42)

New columns added:
['Mjob_health', 'Mjob_other', 'Mjob_services', 'Mjob_teacher', 'Fjob_health', 'Fjob_other', 'Fjob_services', 'Fjob_teacher', 'reason_home', 'reason_other', 'reason_reputation', 'guardian_mother', 'guardian_other']


## Define Targets

We will train models for two tasks:
- **Regression**: predict `G3` as a continuous score (0–20)
- **Classification**: predict `pass` = 1 if G3 ≥ 10, else 0

As `G1` and `G2` are intermidiate grades, we don't really want them in the dataset, so they are dropped in order to only focus for the predictions in `G3`.

In [4]:
def prepare_targets(df):
    df = df.copy()
    df['pass'] = (df['G3'] >= 10).astype(int)
    return df

math_df = prepare_targets(math_df)
por_df  = prepare_targets(por_df)

# Features: drop G1, G2, G3, and pass (targets)
feature_cols = [c for c in math_df.columns if c not in ['G1', 'G2', 'G3', 'pass']]

print(f'{len(feature_cols)} features:')
print(feature_cols)

print('\nPass rate for Math:      ', math_df['pass'].mean().round(3))
print('Pass rate for Portuguese:', por_df['pass'].mean().round(3))

39 features:
['school', 'sex', 'age', 'address', 'famsize', 'Pstatus', 'Medu', 'Fedu', 'traveltime', 'studytime', 'failures', 'schoolsup', 'famsup', 'paid', 'activities', 'nursery', 'higher', 'internet', 'romantic', 'famrel', 'freetime', 'goout', 'Dalc', 'Walc', 'health', 'absences', 'Mjob_health', 'Mjob_other', 'Mjob_services', 'Mjob_teacher', 'Fjob_health', 'Fjob_other', 'Fjob_services', 'Fjob_teacher', 'reason_home', 'reason_other', 'reason_reputation', 'guardian_mother', 'guardian_other']

Pass rate for Math:       0.671
Pass rate for Portuguese: 0.846


In [6]:
math_df.to_csv('./data/math_processed.csv', index=False)
por_df.to_csv('./data/portuguese_processed.csv', index=False)

print('Datasets saved...')

Datasets saved...
